# Implementing DICE paper `Methods for Numeracy-Preserving Word Embeddings` by Sundaraman et al

## 1. Motivation and Problem Statement

### 1.1 The Core Problem
Most popular word embedding models like **word2vec** and **GloVe** to contextual models like BERT are built on the distributional hypothesis. This hypothesis states that a word's meaning is defined by the company it keeps. This works beautifully for semantic concepts (e.g., "cat" and "kitten" appear in similar contexts) but it fails spectacularly for numbers.<br>

It happens basically due to the fundamental properties of numbers. The numbers *3* and *4* might appear in very similar contexts ("I have 3 apples," "I have 4 apples"). Based on context alone a model would learn that their embeddings should be very close.

1. **Magnitude:** 4 is greater than 3.
2. **Numeration:** 3 is the same as "three".

This failure is a major issue and it causes models to fail at any task requiring numerical reasoning such as question answering, sentence similarity and data-intensive tasks like machine translation or mathematical operations.

- **Problem Statement:** The authors state that existing word embedding models are ineffective at capturing the numeric properties of numbers. They treat numbers like any other arbitrary word token which leads to unintuitive similarities and a failure to encode magnitude. The goal is to create a new method for number embeddings that explicitly and systematically captures these numerical properties.


## 2. Assumptions and Goals

### 2.1 Assumptions
The paper's central hypothesis or assumption is that a superior numerical embedding can be created if its structure we are not learning from a corpus but instead building deterministically. Specifically authors propose that the cosine similarity between two number embeddings should directly reflect their actual distance on the number line.

### 2.2 Goals
The primary goal is to develop a method to assign and learn embeddings for numbers that correctly capture their numerical properties. Also we want following goals accomplished:
* Provide embeddings $e(x)$ such that cosine distance $d_{\text{cos}}(e(x), e(y))$ monotonically increases with $|x-y|$.
* Keep numeral and word-form representations identical (word tokens that denote numbers point to the numeral embedding).
* Offer a regularizer usable during contextual fine-tuning that encourages similar behavior for learned contextual embeddings.


## 3. Mathematical and Theoretical Concepts

The paper presents two key mathematical concepts: the DICE embedding itself and the regularization loss for contextual models.

### 3.1 DICE: Deterministic Independent of Corpus Embeddings

The core idea of DICE is to map the distance between numbers to the cosine distance between their embedding vectors.

1. **Number Distance ($d_n$):** In the token space or the real number line the distance between two numbers $x$ and $y$ is their absolute difference i.e. $d_n(x, y) = |x - y|$

2. **Embedding Distance $(d_e​)$:** In the embedding space, the distance between two vectors x and y is their cosine distance which is based on the angle θ between them i.e. $d_e(\mathbf{x}, \mathbf{y}) = 1 - \cos(\theta) = 1 - \frac{\mathbf{x}^T \mathbf{y}}{\|\mathbf{x}\|_2 \|\mathbf{y}\|_2}$

3. **Goal:** We want $d_e$ to increase as $d_n$ increases.

#### Algorithm:
```bash
Inputs: numeric range [a, b], embedding dimension D, set of numbers S (subset of [a,b])
1. For each number x in S:
    1.1 θ = (x - a) / |a - b| * π #maps x to [0, π]
    1.2 Construct v ∈ R^D using spherical coords (Eq.6) with angle θ
2. Sample M ∈ R^{D×D} with iid N(0,1) entries
3. QR-decompose M = Q R  (Q orthonormal basis)
4. For each v, the final embedding e(x) = Q * v #random rotation/projection
5. Normalize embeddings (optional: unit norm)
6. For any word token that spells the number (e.g., "three"), point its embedding to same vector e(3).
```

#### Explanation:
The authors achieve this by mapping each individual number $x$ to a specific vector $\mathbf{e}_x$.
- **Angle Mapping:** First authors define a range of numbers they care about $[a, b]$. They linearly map any number $x$ in this range to an angle $\theta(x)$ between $0$ and $\pi$. A number is thus defined by this angle.

- **Vector Generation:** This single angle $\theta$ is then used to generate a $D$-dimensional unit vector $\mathbf{v}$ using a $D$-dimensional polar to cartesian (hyperspherical) coordinate transformation.

- **Random Rotation:** This vector $\mathbf{v}$ is then rotated in the $D$-dimensional space. This is done by multiplying it by a fixed random $D \times D$ orthonormal matrix $\mathbf{Q}$ (which is obtained from a QR decomposition of a random matrix $\mathbf{M}$).

- **Final Embedding:** The final embedding for number $x$ is $\mathbf{e}_x = \mathbf{Q}\mathbf{v}(x)$. This embedding is deterministic (once $\mathbf{Q}$ is fixed) and corpus-independent (it only depends on the value of $x$).

#### Example:
Let $[a,b]=[0,100]$ and $(D=2)$

* For $x=25$: $\theta(25) = 25/100 \cdot \pi = 0.25\pi = 45^\circ$.<br>

  Embedding in 2D: $[ \cos(45^\circ), \sin(45^\circ) ] = [\tfrac{\sqrt2}{2}, \tfrac{\sqrt2}{2}]$.

* For $y=75$: $\theta(75) = 0.75\pi = 135^\circ)$ <br>

  Embedding: $[-\tfrac{\sqrt2}{2}, \tfrac{\sqrt2}{2}]$

* Angle between them is $90^\circ$ and cosine similarity is 0 with cosine distance 1 which tells us that they are mid-range apart $|25−75|=50$.

### 3.2 $ \mathcal{L}_{num}$: Model Based Numeracy Regularization

For contextual models like BERT the authors propose an auxiliary loss function, $\mathcal{L}_{num}$ which is to be added during fine-tuning.<br>
The loss forces the model to learn numerically aware embeddings in context. For any two number embeddings $\mathbf{x}$ and $\mathbf{y}$ taken from the model's final hidden layer then the loss is $\mathcal{L}_{num} = \left\| d_{target} - d_{cos}(\mathbf{x}, \mathbf{y}) \right\|_2^2$

- $\mathbf{x}, \mathbf{y}$ are the contextual embeddings for numbers $x$ and $y$.

- $d_{cos}(\mathbf{x}, \mathbf{y})$ is their cosine distance.

- $d_{target} = \frac{2 |x-y|}{|x|+|y|}$ is the target distance which is a normalized numerical distance between $x$ and $y$ which is always between 0 and 1.

This loss function teaches BERT that the cosine distance between its embeddings for **5** and **10** should be smaller than the distance between its embeddings for **5** and **100**.

## 4. Implementation in Python

### 4.1 Importing Dependencies 

In [1]:
import re
import os
import h5py
import json
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
from datetime import datetime
import matplotlib.pyplot as plt
from collections import defaultdict

from sklearn.decomposition import PCA
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from typing import List, Tuple, Union, Dict, Optional 
from scipy.spatial.distance import cosine as cosine_dist
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.metrics.pairwise import cosine_distances, cosine_similarity



warnings.filterwarnings('ignore')
np.random.seed(42)

### 4.2 Core DICE Implementation

In [ ]:
class DICEEmbeddings:
    
    """
    Here we define the DICEEmbeddings class which will handle the creation of DICE embeddings.
    dimension: Dimension of the embedding space.
    number_range: Range of numbers to be embedded.
    random_rotation: Whether to apply random rotation to the embeddings.
    seed: Random seed for reproducibility.
    """
    def __init__(self, dimension: int = 300, number_range: Tuple[float, float] = (0, 10000),
                 random_rotation: bool = True, seed: int = 42):
        self.dimension = dimension
        self.a, self.b = number_range# a and b are the min and max numbers
        self.range_size = self.b - self.a# Range size
        self.seed = seed
        
        #preventing division by zero
        if self.range_size == 0:
            self.range_size = 1e-9# Small value to prevent division by zero
        
        self.random_rotation = random_rotation# Whether to apply random rotation
        #generating random rotation matrix using QR decomposition as described in the paper
        if self.random_rotation:
            np.random.seed(self.seed)# Seed for reproducibility
            random_matrix = np.random.randn(self.dimension, self.dimension)# Random matrix
            self.rotation_matrix, _ = np.linalg.qr(random_matrix)# QR decomposition to get orthogonal matrix where _ means we ignore the second output
        else:
            self.rotation_matrix = np.eye(self.dimension)# Identity matrix if no rotation
        
        self._embedding_cache = {}# Cache for embeddings to speed up repeated computations
    
    """
    this function computes the spherical coordinates for a given angle theta.
    theta: Angle in radians.
    returns: Numpy array of spherical coordinates.
    v_d = [sin(θ)]^(d-1) * cos(θ)  for 1 ≤ d < D
    v_D = [sin(θ)]^D               for d = D
    """
    def _spherical_coordinates(self, theta:float) -> np.ndarray:
        v = np.zeros(self.dimension)# initialising vector
        sin_theta = np.sin(theta)# sin(θ)
        
        sin_power = 1.0# sin(θ)^(d-1) for avoidance of repeated computation because sin(θ) is used multiple times
        for d in range(1, self.dimension+1):
            if d < self.dimension:# d < D
                # v_d = [sin(θ)]^(d-1) * cos(θ)
                v[d-1] = sin_power * np.cos(theta)
                sin_power *= sin_theta# updating sin(θ)^(d-1) to sin(θ)^d for next iteration
            else:# d = D
                # v_D = [sin(θ)]^D
                v[d-1] = sin_power * sin_theta# sin(θ)^D
        return v
    
    """
    This function maps a number to an angle in radians within the range [0, π/2].
    θ(s_n) = (s_n / |a - b|) * π
    where s_n is the distance from the lower bound a.
    number: The number to be mapped.
    returns: Angle in radians.
    """
    def _number_to_angle(self, number: float) -> float:
        if number < self.a or number > self.b:
            np.random.seed(int(hash(number) % (2**32)))
            theta = np.random.uniform(-np.pi, np.pi)
        else:
            # Linear mapping from [a, b] to [0, π]
            normalized_distance = (number - self.a) / self.range_size
            theta = normalized_distance * np.pi

        return theta
    
    """
    This function computes the DICE embedding for a given number.
    number: The number to be embedded.
    use_cache: Whether to use cached embeddings.
    returns: Numpy array of the DICE embedding.
    """
    def get_embedding(self, number: Union[int, float], use_cache: bool = True) -> np.ndarray:
        # Check cache
        if use_cache and number in self._embedding_cache:
            return self._embedding_cache[number]
        
        # Map number to angle
        theta = self._number_to_angle(number)
        
        # Get spherical coordinate vector
        v = self._spherical_coordinates(theta)
        
        # Apply random rotation (Q @ v)
        embedding = self.rotation_matrix @ v
        
        # Cache the result
        if use_cache:
            self._embedding_cache[number] = embedding
        
        return embedding
    
    """
    This function computes DICE embeddings for a batch of numbers.
    numbers: List of numbers to be embedded.
    use_cache: Whether to use cached embeddings.
    returns: Numpy array of DICE embeddings.
    """
    def get_embeddings_batch(self, numbers: List[Union[int, float]], 
                            use_cache: bool = True) -> np.ndarray:
        return np.array([self.get_embedding(num, use_cache) for num in numbers])
    
    def clear_cache(self):# Clear the embedding cache
        self._embedding_cache = {}
    
    # This function retrieves the angle corresponding to a given number.
    def get_angle(self, number: Union[int, float]) -> float:
        return self._number_to_angle(number)